In [2]:
import numpy as np
import tensorflow as tf

from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Dense

# -----------------------------
# 1. Fix randomness (important)
# -----------------------------
np.random.seed(42)
tf.random.set_seed(42)

# -----------------------------
# 2. Create Dummy Dataset
# -----------------------------
num_samples = 10000
max_len = 5
vocab_size = 10

# Input sequences
X = np.random.randint(1, vocab_size, size=(num_samples, max_len))

# Output = reversed sequence
y = np.flip(X, axis=1)

# One-hot encoding
X = tf.keras.utils.to_categorical(X, num_classes=vocab_size)
y = tf.keras.utils.to_categorical(y, num_classes=vocab_size)

# -----------------------------
# 3. Encoder
# -----------------------------
encoder_inputs = Input(shape=(None, vocab_size))
encoder_lstm = LSTM(64, return_state=True)

encoder_outputs, state_h, state_c = encoder_lstm(encoder_inputs)

encoder_states = [state_h, state_c]

# -----------------------------
# 4. Decoder
# -----------------------------
decoder_inputs = Input(shape=(None, vocab_size))

decoder_lstm = LSTM(64, return_sequences=True, return_state=True)

decoder_outputs, _, _ = decoder_lstm(
    decoder_inputs,
    initial_state=encoder_states
)

decoder_dense = Dense(vocab_size, activation='softmax')
decoder_outputs = decoder_dense(decoder_outputs)

# -----------------------------
# 5. Seq2Seq Model
# -----------------------------
model = Model([encoder_inputs, decoder_inputs], decoder_outputs)

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

# -----------------------------
# 6. Train Model
# -----------------------------
history = model.fit(
    [X, X],   # teacher forcing
    y,
    epochs=5,
    batch_size=64,
    validation_split=0.2
)

# -----------------------------
# 7. Test Prediction
# -----------------------------
sample_input = X[:1]

prediction = model.predict([sample_input, sample_input])

predicted_seq = np.argmax(prediction, axis=-1)
original_seq = np.argmax(sample_input, axis=-1)

print("\nOriginal Sequence :", original_seq)
print("Predicted Reverse :", predicted_seq)

2026-05-04 19:50:19.722255: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, None, 10)  │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_1       │ (None, None, 10)  │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm (LSTM)         │ [(None, 64),      │     19,200 │ input_layer[0][0] │
│                     │ (None, 64),       │            │                   │
│                     │ (None, 64)]       │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_1 (LSTM)       │ [(None, None,     │     19,200 │ input_layer_1[0]… │
│                     │ 64), (None, 64),  │            │ lstm[0][1],       │
│                     │ (None, 64)]       │            │ lstm[0][2]        │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, None, 10)  │        650 │ lstm_1[0][0]      │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 39,050 (152.54 KB)

 Trainable params: 39,050 (152.54 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/5
125/125 ━━━━━━━━━━━━━━━━━━━━ 7s 15ms/step - accuracy: 0.3201 - loss: 1.8259 - val_accuracy: 0.3750 - val_loss: 1.4906
Epoch 2/5
125/125 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.4066 - loss: 1.3763 - val_accuracy: 0.4650 - val_loss: 1.2352
Epoch 3/5
125/125 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.5283 - loss: 1.0827 - val_accuracy: 0.5999 - val_loss: 0.9421
Epoch 4/5
125/125 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.6612 - loss: 0.8196 - val_accuracy: 0.7246 - val_loss: 0.7086
Epoch 5/5
125/125 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.7783 - loss: 0.6009 - val_accuracy: 0.8178 - val_loss: 0.5173
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 438ms/step

Original Sequence : [[7 4 8 5 7]]
Predicted Reverse : [[5 5 4 4 4]]
